# Galaxy10 DECaLS: Basic Fine-tuning Tutorial

**Project:** *Faking Galaxies with Diffusion: A Real-vs-Synthetic Benchmark for Morphology Classification*
Hunter Sandoval. Course project, University of New Mexico. Tutorial 5 of 6.

**Scope.** Both of the project's models are *fine-tuned* versions of pretrained models. This tutorial downloads the
pretrained models, sets up the project's optimization loops, and retrains them for a short while to show the loss going
down:

1. **Classifier**: torchvision's ImageNet-pretrained ResNet-18, fine-tuned into a 10-class galaxy classifier;
2. **Generator**: Stable Diffusion 1.5 from Hugging Face, fine-tuned with LoRA adapters and a learned class-embedding
   table into a class-conditional galaxy generator (a few galaxies are generated before and after).

**FAST mode (on by default)** shrinks the runs so the whole notebook finishes in a few minutes on a free Colab T4: a
fixed-seed 10% stratified subset of the training images, 5 classifier epochs, 200 LoRA steps at batch 4, and 4
generated images. `FAST = False` runs the paper's full settings (table below). The **final results cells** load the
project's released weights, i.e. the models trained with the full settings, so you can compare.

## References

1. K. He et al., "Deep Residual Learning for Image Recognition," CVPR 2016, arXiv:1512.03385; torchvision's ImageNet
   weights: https://pytorch.org/vision/stable/models/generated/torchvision.models.resnet18.html
2. R. Rombach et al., "High-Resolution Image Synthesis with Latent Diffusion Models," CVPR 2022, arXiv:2112.10752;
   weights: https://huggingface.co/stable-diffusion-v1-5/stable-diffusion-v1-5
3. E. J. Hu et al., "LoRA: Low-Rank Adaptation of Large Language Models," ICLR 2022, arXiv:2106.09685.
4. J. Ho, A. Jain, P. Abbeel, "Denoising Diffusion Probabilistic Models," NeurIPS 2020, arXiv:2006.11239.
5. J. Ho and T. Salimans, "Classifier-Free Diffusion Guidance," arXiv:2207.12598, 2022.
6. I. Loshchilov and F. Hutter, "Decoupled Weight Decay Regularization," ICLR 2019, arXiv:1711.05101 (AdamW).
7. P. Micikevicius et al., "Mixed Precision Training," ICLR 2018, arXiv:1710.03740 (fp16 with loss scaling).
8. Hugging Face diffusers LoRA training example (the structure the project's `diffusion/train_lora.py` follows):
   https://github.com/huggingface/diffusers/blob/main/examples/text_to_image/train_text_to_image_lora.py
9. Project code and model weights: https://github.com/HunterSandoval12/faking-galaxies-with-diffusion
10. Course tutorial requirements: https://github.com/pattichis/projects/blob/main/Colab-tutorial-list.md

# Setup

## Library requirements and installation

| Library | Minimum version | Tested version | Install command | Used for |
|---|---|---|---|---|
| Python | 3.10 | 3.14.7 | (Colab's default) | |
| PyTorch (`torch`) | 2.0 | 2.14.0 | `pip install "torch>=2.0"` | training |
| torchvision | 0.15 | 0.29.0 | `pip install "torchvision>=0.15"` | ResNet-18 and its ImageNet weights |
| diffusers | 0.27 | 0.40.0 | `pip install "diffusers>=0.27"` | Stable Diffusion's VAE, U-Net, schedulers |
| peft | 0.10 | 0.21.0 | `pip install "peft>=0.10"` | LoRA adapters |
| transformers | 4.30 | 5.17.0 | `pip install "transformers>=4.30"` | the CLIP text encoder (initial class table) |
| safetensors | 0.4 | 0.8.0 | `pip install "safetensors>=0.4"` | reading the released weights |
| NumPy | 1.23 | 2.5.2 | `pip install "numpy>=1.23"` | arrays |
| Pillow | 9.0 | 12.3.0 | `pip install "pillow>=9.0"` | images |
| Matplotlib | 3.6 | 3.11.2 | `pip install "matplotlib>=3.6"` | figures |

Google Colab preinstalls all of these, so the next cell normally installs nothing. It checks each library and runs
`pip` only for a missing or too-old one. On a **second run**, set `SKIP_INSTALL = True` to skip the check entirely.

**One conflict to remove.** Colab also preinstalls an old version of `torchao` (a quantization library this tutorial
does not use). Recent `peft` versions refuse to create LoRA adapters while a `torchao` older than 0.16 is installed
("ImportError: Found an incompatible version of torchao"), so the cell uninstalls such an old `torchao`
(`pip uninstall -y torchao`). This check runs even when `SKIP_INSTALL = True`.

In [ ]:
SKIP_INSTALL = False  # set to True when re-running the notebook to skip the installation check

import importlib.metadata
import re
import subprocess
import sys

REQUIREMENTS = {"torch": "2.0",
                "torchvision": "0.15",
                "diffusers": "0.27",
                "peft": "0.10",
                "transformers": "4.30",
                "safetensors": "0.4",
                "numpy": "1.23",
                "pillow": "9.0",
                "matplotlib": "3.6"}  # package -> minimum version


def version_tuple(v):
    return tuple(int(x) for x in re.findall(r"\d+", v)[:3])


def installed_version(pkg):
    try:
        return importlib.metadata.version(pkg)
    except importlib.metadata.PackageNotFoundError:
        return None


if SKIP_INSTALL:
    print("SKIP_INSTALL = True: skipping the installation check.")
else:
    to_install = [f"{pkg}>={minimum}" for pkg, minimum in REQUIREMENTS.items()
                  if installed_version(pkg) is None or version_tuple(installed_version(pkg)) < version_tuple(minimum)]
    if to_install:
        print("Installing:", to_install)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *to_install])
        print("Done. If a library was upgraded, restart the runtime (Runtime > Restart session) and run again.")
    else:
        print("All required libraries are already installed; nothing to install.")

# peft (LoRA) raises an ImportError while a torchao older than 0.16 is installed; this notebook does not use torchao.
TORCHAO_MIN = "0.16"
old_torchao = installed_version("torchao")
if old_torchao is not None and version_tuple(old_torchao) < version_tuple(TORCHAO_MIN):
    print(f"Removing torchao {old_torchao} (incompatible with peft; not used here)")
    subprocess.check_call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"])
    import importlib
    importlib.invalidate_caches()  # let this running session see that torchao is gone

In [ ]:
# Verify the versions. The notebook stops here with a clear message if a requirement is not met.
assert sys.version_info >= (3, 10), f"Python >= 3.10 required, found {sys.version.split()[0]}"
torchao = installed_version("torchao")
assert torchao is None or version_tuple(torchao) >= version_tuple(TORCHAO_MIN), \
    f"torchao {torchao} is installed and breaks peft: run !pip uninstall -y torchao"
for pkg, minimum in REQUIREMENTS.items():
    have = installed_version(pkg)
    assert have is not None, f"{pkg} is not installed (pip install '{pkg}>={minimum}')"
    assert version_tuple(have) >= version_tuple(minimum), f"{pkg} {have} is older than the required {minimum}"
    print(f"{pkg:<13} {have:<14} (required >= {minimum})")
print("All version requirements are satisfied.")

## Hardware used for running

- **GPU required**: in Colab choose *Runtime > Change runtime type > T4 GPU* (the notebook asks for it when opened from
  GitHub). The free T4 has 15 GB of memory and no bf16 hardware, so the notebook trains in fp16 there (next section).
- **Peak GPU memory (FAST mode, fp16):** 2.2 GB for the classifier and 6.3 GB for the LoRA fine-tuning,
  measured with the GPU limited to 15 GB like a T4. Full settings (`FAST = False`): the LoRA training at batch 8
  needs 10.6 GB in fp16 (measured the same way), so it fits a T4 too.
- **Tested on:** a Windows 11 PC with an NVIDIA RTX 5090 (32 GB), AMD Ryzen 7 9800X3D, 32 GB RAM, Python 3.14. Every
  code cell was run there in fp16 with GPU memory capped at 15 GB (to mimic the T4) and in the GPU's native bf16.

## Approximate execution times

| Step | Test PC (FAST, fp16) | Colab T4 (FAST, estimate) | Full settings on the RTX 5090 (the paper's runs) |
|---|---|---|---|
| Load the images from Tutorial 1's zip | ~5 s | 1-2 min | - |
| Classifier fine-tuning | ~10 s | ~1-2 min | 1.5 min per run (30 epochs on 12,330 images, bf16, incl. per-epoch validation) |
| LoRA fine-tuning (incl. model download) | ~42 s | ~5-8 min | 60 min for 10,000 steps at batch 8 (0.36 s/step, bf16) |
| Generating 12 images (3 x 4) | ~8 s | ~2 min | - |
| **Whole tutorial** | **~1.2 min** | **~12-18 min** | - |

The Colab column is an estimate: for these models a T4 is roughly 8-10 times slower than the RTX 5090. With the full
settings (`FAST = False`) a T4 would need about 15 min for the classifier and 8-10 hours for the LoRA fine-tuning.

## Expected results

- **Classifier**: before fine-tuning (new random output layer) the validation loss is about 2.3 = ln 10 and the
  accuracy ~10%. After 5 FAST epochs on 10% of the data the training loss has dropped steadily and the validation
  accuracy is about 71% in fp16 (70% in bf16). The released classifier (30 epochs, all data) reaches 86.7% validation accuracy.
- **Generator**: the per-step diffusion loss is noisy and nearly flat (by design, see Tutorial 3); the loss on a fixed
  evaluation batch decreases slightly (0.1923 -> 0.1904, about -1%). The generated galaxies change from Stable Diffusion's idea of a
  "telescope image of a galaxy" towards the survey's look; the released generator (10,000 steps) produces the images of
  the paper's synthetic set.

# Mount Google Drive

The fine-tuning uses the training and validation images from Tutorial 1. Tutorial 1 (Dataset Tutorial) saved the `training` / `validation` / `testing` directories on your Google
Drive as `MyDrive/galaxy10_tutorial/galaxy10_dataset.zip`. This tutorial reads single images straight out of that zip
file with Python's `zipfile` module, so nothing needs to be unzipped. Mounting makes your Drive a folder of the Colab
machine:

```python
from google.colab import drive
drive.mount("/content/drive")   # asks you to authorise access the first time
```

Outside Colab, the cell uses the same local folders as Tutorial 1 (`./galaxy10_tutorial_drive` and
`./galaxy10_tutorial_work`, or the environment variables `GALAXY10_TUTORIAL_DRIVE` / `GALAXY10_TUTORIAL_WORK`).

In [ ]:
import os
import time
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
T0 = time.time()
TIMES = {}  # execution time of each step (seconds), reported at the end

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    DRIVE_ROOT = Path("/content/drive/MyDrive/galaxy10_tutorial")  # persistent (Google Drive)
    WORK_ROOT = Path("/content")                                      # fast, wiped after the session
else:
    DRIVE_ROOT = Path(os.environ.get("GALAXY10_TUTORIAL_DRIVE", "galaxy10_tutorial_drive")).resolve()
    WORK_ROOT = Path(os.environ.get("GALAXY10_TUTORIAL_WORK", "galaxy10_tutorial_work")).resolve()
ZIP_PATH = DRIVE_ROOT / "galaxy10_dataset.zip"  # written by Tutorial 1
WEIGHTS_DIR = WORK_ROOT / "weights"             # downloaded model files
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
print("Running in Colab:", IN_COLAB)
print("Dataset zip from Tutorial 1:", ZIP_PATH, "(found)" if ZIP_PATH.exists() else "(NOT FOUND)")

In [ ]:
import io
import zipfile

from PIL import Image

CLASS_NAMES = ["Disturbed", "Merging", "Round Smooth", "In-between Round Smooth", "Cigar-Shaped Smooth",
               "Barred Spiral", "Unbarred Tight Spiral", "Unbarred Loose Spiral", "Edge-on without Bulge",
               "Edge-on with Bulge"]
assert ZIP_PATH.exists(), "Run Tutorial 1 (Dataset Tutorial) first: it saves galaxy10_dataset.zip on your Drive."
dataset_zip = zipfile.ZipFile(ZIP_PATH)


def zip_members(archive, folder):
    """[(member, label)] of <root>/<folder>/<label>_<class>/<file>.png in a zip, sorted by class and file name."""
    out = []
    for n in sorted(archive.namelist()):
        parts = n.split("/")
        if n.endswith(".png") and len(parts) == 4 and parts[1] == folder:
            out.append((n, int(parts[2].split("_")[0])))
    return out


def stratified(items, fraction, seed=0):
    """Fixed-seed stratified subset: the same fraction of every class (at least 1 image per class)."""
    if fraction >= 1:
        return items
    rng = np.random.default_rng(seed)
    labels = np.array([c for _, c in items])
    keep = []
    for c in range(len(CLASS_NAMES)):
        idx = np.flatnonzero(labels == c)
        keep += rng.choice(idx, max(1, round(fraction * len(idx))), replace=False).tolist()
    return [items[i] for i in sorted(keep)]


def load_images(archive, items):
    """uint8 images [N, 256, 256, 3] and int64 labels [N] as (CPU) torch tensors."""
    x = np.stack([np.asarray(Image.open(io.BytesIO(archive.read(m))).convert("RGB")) for m, _ in items])
    return torch.from_numpy(x), torch.tensor([c for _, c in items], dtype=torch.int64)

# FAST mode and the full settings

| Setting | FAST (default) | Full (paper) |
|---|---|---|
| Classifier training images | 10% stratified subset (1,234; fixed seed) | all 12,330 |
| Classifier epochs | 5 | 30 |
| Classifier batch / learning rate | 128 / 1e-3 (1 warm-up epoch + cosine) | 128 / 1e-3 (1 warm-up epoch + cosine) |
| Validation images | all 2,607 | all 2,607 |
| LoRA training images | the same 10% subset | all 12,330 |
| LoRA steps / batch | 200 / 4 | 10,000 / 8 |
| LoRA warm-up steps | 20 | 500 |
| Generated images | 4 classes x (before, after, released) | - |
| Time on the RTX 5090 | minutes | classifier 1.5 min per run (30 epochs on 12,330 images, bf16, incl. per-epoch validation); LoRA 60 min for 10,000 steps at batch 8 (0.36 s/step, bf16) |

In [ ]:
FAST = True  # False: the paper's full settings (hours on a T4)

if FAST:
    TRAIN_FRACTION, CLS_EPOCHS = 0.1, 5
    LORA_STEPS, LORA_BATCH, LORA_WARMUP, LORA_EVAL_EVERY = 200, 4, 20, 25
else:
    TRAIN_FRACTION, CLS_EPOCHS = 1.0, 30
    LORA_STEPS, LORA_BATCH, LORA_WARMUP, LORA_EVAL_EVERY = 10000, 8, 500, 500
GEN_CLASSES = [2, 4, 5, 9]  # Round Smooth, Cigar-Shaped Smooth, Barred Spiral, Edge-on with Bulge
GEN_SEEDS = [7_000_000_000 + c * 1_000_000 for c in GEN_CLASSES]  # the paper's seeds for sample 0 of each class
print(f"FAST = {FAST}: {TRAIN_FRACTION:.0%} of the training images, {CLS_EPOCHS} classifier epochs, "
      f"{LORA_STEPS} LoRA steps at batch {LORA_BATCH}")

# Download the released weights (for the final results)

The fine-tuning below starts from the *public pretrained* models (ImageNet ResNet-18, Stable Diffusion 1.5). The project's released weights, trained with the full settings, are downloaded here only for the final comparison cells.
The files are attached to the project's GitHub Release `weights-v1`. A release file has a fixed, public address,
`https://github.com/HunterSandoval12/faking-galaxies-with-diffusion/releases/download/weights-v1/<file name>`, so plain `wget` downloads it; no GitHub account or login is needed. After each download
the cell checks the file's **SHA-256 checksum** against the value published in the repository's README and in the
release notes, so a corrupted or altered file stops the notebook:

| File | SHA-256 |
|---|---|
| `resnet18_real_only_seed0.safetensors` | `ad9a4e605ef50154a6f3e377c23b4f95fae3ca6712b9a5815b4dbfcdc66f40dd` |
| `lora_unet.safetensors` | `9b866502137e2b9ac7783f84b9a8110578a51aeb60c93779e03627bb0faeb20a` |
| `class_embeddings.safetensors` | `87a7fb2720dc4a0dedb66c3417265b63d02ff7b829ea86465dfc11ee2468a2cb` |
| `lora_config.json` | `29dfa782205ee5875677d8cfb358c08c8aab8290442fb1526abcab5ba7b8baf1` |

**How we figured this out.** GitHub serves every release file at `.../releases/download/<tag>/<file name>` and
redirects the request to its file storage; `wget` follows the redirect. `hashlib.sha256` computes the checksum in
Python (on Linux, `sha256sum <file>` gives the same value).

In [ ]:
import hashlib

RELEASE_URL = "https://github.com/HunterSandoval12/faking-galaxies-with-diffusion/releases/download/weights-v1"
FILES = {  # file name -> SHA-256 checksum
    "resnet18_real_only_seed0.safetensors": "ad9a4e605ef50154a6f3e377c23b4f95fae3ca6712b9a5815b4dbfcdc66f40dd",
    "lora_unet.safetensors": "9b866502137e2b9ac7783f84b9a8110578a51aeb60c93779e03627bb0faeb20a",
    "class_embeddings.safetensors": "87a7fb2720dc4a0dedb66c3417265b63d02ff7b829ea86465dfc11ee2468a2cb",
    "lora_config.json": "29dfa782205ee5875677d8cfb358c08c8aab8290442fb1526abcab5ba7b8baf1",
}


def sha256_of(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        while block := f.read(2**24):
            h.update(block)
    return h.hexdigest()


t = time.time()
for name, expected in FILES.items():
    path = WEIGHTS_DIR / name
    if not (path.exists() and sha256_of(path) == expected):  # skip files downloaded earlier
        url = f"{RELEASE_URL}/{name}"
        !wget -q -O "{path}" "{url}"
    assert path.exists() and sha256_of(path) == expected, \
        f"{name}: download failed or checksum mismatch; delete {path} and run this cell again"
    print(f"{name:<42} {path.stat().st_size / 2**20:6.2f} MiB   SHA-256 OK")
TIMES["download weights"] = time.time() - t

## GPU, precision and memory

Training runs in **mixed precision**: the heavy matrix products use 16-bit numbers, the trained weights stay in 32 bits.
There are two 16-bit formats. **bf16** has the same range as 32-bit floats and needs no extra care, but only GPUs with
compute capability 8.0 or newer (Ampere and later, e.g. the A100, L4 or the project's RTX 5090) have bf16 hardware.
Colab's free **T4** is a Turing GPU (compute capability 7.5): it has fast **fp16** tensor cores but no bf16 hardware,
so bf16 would be emulated and slow. `torch.cuda.is_bf16_supported()` can still report `True` on a T4, because the
emulation *works*; the cell therefore checks the compute capability itself.

With fp16, small gradients can underflow to zero, so fp16 training uses a **gradient scaler**
(`torch.amp.GradScaler`): the loss is multiplied by a large factor before backpropagation and the gradients are divided
by it again before the optimizer step; a step whose gradients overflow is skipped and the factor lowered. The project's
own runs used bf16 on the RTX 5090, so numbers from fp16 runs differ slightly.

`PRECISION = "auto"` picks bf16 when the GPU has it and fp16 otherwise; `"fp16"`, `"bf16"` or `"fp32"` force one.
After each training stage the notebook reports the peak GPU memory, to show that it fits the T4's 15 GB.

In [ ]:
import numpy as np
import torch

PRECISION = "auto"  # "auto" | "bf16" | "fp16" | "fp32"

assert torch.cuda.is_available(), "This tutorial needs a GPU: in Colab choose Runtime > Change runtime type > T4 GPU"
DEVICE = torch.device("cuda")
props = torch.cuda.get_device_properties(0)
NATIVE_BF16 = props.major >= 8  # bf16 tensor cores: Ampere (8.x) and newer
AMP_DTYPE = {"auto": torch.bfloat16 if NATIVE_BF16 else torch.float16, "bf16": torch.bfloat16,
             "fp16": torch.float16, "fp32": torch.float32}[PRECISION]
USE_GRAD_SCALER = AMP_DTYPE == torch.float16  # only fp16 needs loss scaling
GPU_GIB = props.total_memory / 2**30
MEMORY = {}  # peak GPU memory per stage (GiB)


def memory_report(stage):
    """Record and print the peak GPU memory since the last report."""
    peak = torch.cuda.max_memory_allocated() / 2**30
    MEMORY[stage] = peak
    print(f"[{stage}] peak GPU memory {peak:.1f} GiB of {GPU_GIB:.1f} GiB")
    torch.cuda.reset_peak_memory_stats()


print(f"GPU: {props.name}, compute capability {props.major}.{props.minor}, {GPU_GIB:.1f} GiB")
print(f"native bf16 hardware: {NATIVE_BF16}; torch.cuda.is_bf16_supported(): {torch.cuda.is_bf16_supported()}")
print(f"training precision: {str(AMP_DTYPE).replace('torch.', '')}; gradient scaler: {USE_GRAD_SCALER}")

# Part 1: Fine-tuning the classifier

## Load the training and validation images

The training images are a fixed-seed **stratified** subset: the same fraction of every class, so the class imbalance
of the full set (e.g. only 233 Cigar-Shaped Smooth images) is kept. All validation images are used for evaluation.

In [ ]:
t = time.time()
train_items = stratified(zip_members(dataset_zip, "training"), TRAIN_FRACTION, seed=0)
train_x, train_y = load_images(dataset_zip, train_items)
val_x, val_y = load_images(dataset_zip, zip_members(dataset_zip, "validation"))
TIMES["load images"] = time.time() - t
print(f"training: {len(train_x)} images, per class {np.bincount(train_y.numpy(), minlength=10).tolist()}")
print(f"validation: {len(val_x)} images; loaded in {TIMES['load images']:.0f} s")

## The pretrained model and the optimization loop

**Download the pretrained model.** `resnet18(weights=ResNet18_Weights.IMAGENET1K_V1)` downloads torchvision's
ImageNet-trained weights (45 MB, from PyTorch's servers). Its 1,000-class output layer is replaced by a new, randomly
initialised 10-class layer; everything else starts from the ImageNet features.

**The optimization loop** is the project's `classifier/train_classifier.py`, reproduced below: every step takes a
batch of 128 images, applies random flips and 90-degree rotations, runs the network in mixed precision, computes the
cross-entropy loss, backpropagates (through the gradient scaler when training in fp16), takes an AdamW step (learning
rate 1e-3, weight decay 0.05) and updates the learning rate (one warm-up epoch, then cosine decay). After every epoch
it measures the validation loss and accuracy and remembers the best epoch (Tutorial 3 explains each choice).

In [ ]:
import copy
import math

import torch.nn as nn
import torch.nn.functional as F
from torchvision.models import ResNet18_Weights, resnet18

IMAGENET_MEAN = torch.tensor([0.485, 0.456, 0.406], device=DEVICE).view(1, 3, 1, 1)
IMAGENET_STD = torch.tensor([0.229, 0.224, 0.225], device=DEVICE).view(1, 3, 1, 1)


def to_input(images_u8, augment):
    """uint8 NHWC on the GPU -> normalised float NCHW; optional random flip + 90-degree rotation (as in the project)."""
    x = images_u8.permute(0, 3, 1, 2).float().div_(255.0)
    if augment:
        flip = torch.rand(x.shape[0], device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(-1), x)
        k = torch.randint(0, 4, (x.shape[0],), device=x.device)
        x = torch.stack([torch.rot90(xi, int(ki), dims=(-2, -1)) for xi, ki in zip(x, k)])
    return (x - IMAGENET_MEAN) / IMAGENET_STD


def build_classifier(pretrained=True):
    """torchvision ResNet-18 (ImageNet weights, downloaded on first use) with a new 10-class output layer."""
    model = resnet18(weights=ResNet18_Weights.IMAGENET1K_V1 if pretrained else None)
    model.fc = nn.Linear(model.fc.in_features, len(CLASS_NAMES))
    return model.to(DEVICE).to(memory_format=torch.channels_last)


@torch.no_grad()
def evaluate(model, x_u8, y, batch_size=256):
    """Mean cross-entropy loss and accuracy of a model on uint8 images."""
    model.eval()
    loss, correct = 0.0, 0
    for s in range(0, len(x_u8), batch_size):
        xb = to_input(x_u8[s:s + batch_size].to(DEVICE), augment=False).contiguous(memory_format=torch.channels_last)
        yb = y[s:s + batch_size].to(DEVICE)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
            logits = model(xb).float()
        loss += F.cross_entropy(logits, yb, reduction="sum").item()
        correct += (logits.argmax(1) == yb).sum().item()
    return loss / len(x_u8), correct / len(x_u8)


def train_classifier(train_x, train_y, eval_sets, epochs, seed, augment=True, learning_rate=1e-3,
                     weight_decay=0.05, batch_size=128, warmup_epochs=1.0, verbose=True):
    """The project's classifier training (classifier/train_classifier.py): AdamW, linear warm-up for one epoch then
    cosine decay to 0 (per step), cross-entropy, mixed precision. After every epoch it evaluates every set in
    eval_sets ({name: (images, labels)}); the returned model is the epoch with the best VALIDATION accuracy.
    Returns (model, per-epoch history, per-step training losses, selected epoch)."""
    torch.manual_seed(seed)
    np.random.seed(seed)
    model = build_classifier()
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)
    steps_per_epoch = math.ceil(len(train_x) / batch_size)
    total, warmup = epochs * steps_per_epoch, int(warmup_epochs * steps_per_epoch)

    def lr_factor(step):
        if step < warmup:
            return (step + 1) / warmup
        return 0.5 * (1 + math.cos(math.pi * (step - warmup) / max(1, total - warmup)))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
    history, step_losses, best_acc, best_state, best_epoch = [], [], -1.0, None, 0
    t0 = time.time()
    for epoch in range(1, epochs + 1):
        model.train()
        perm = torch.randperm(len(train_x))
        loss_sum, correct = 0.0, 0
        for s in range(0, len(perm), batch_size):
            bi = perm[s:s + batch_size]
            x = to_input(train_x[bi].to(DEVICE, non_blocking=True), augment).contiguous(memory_format=torch.channels_last)
            y = train_y[bi].to(DEVICE)
            with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
                logits = model(x)
            loss = F.cross_entropy(logits.float(), y)
            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            step_losses.append(loss.item())
            loss_sum += loss.item() * len(bi)
            correct += (logits.argmax(1) == y).sum().item()
        rec = {"epoch": epoch, "train_loss": loss_sum / len(train_x), "train_accuracy": correct / len(train_x),
               "lr": scheduler.get_last_lr()[0]}
        for name, (ex, ey) in eval_sets.items():
            rec[f"{name}_loss"], rec[f"{name}_accuracy"] = evaluate(model, ex, ey)
        rec["elapsed_s"] = time.time() - t0
        history.append(rec)
        if rec["validation_accuracy"] > best_acc:  # model selection on VALIDATION only
            best_acc, best_epoch = rec["validation_accuracy"], epoch
            best_state = copy.deepcopy(model.state_dict())
        if verbose:
            print(f"  epoch {epoch:>2}/{epochs}  train loss {rec['train_loss']:.3f} acc {rec['train_accuracy']:.3f}  "
                  + "  ".join(f"{n} loss {rec[f'{n}_loss']:.3f} acc {rec[f'{n}_accuracy']:.3f}" for n in eval_sets)
                  + f"  ({rec['elapsed_s']:.0f} s)")
    model.load_state_dict(best_state)
    return model, history, step_losses, best_epoch

## Before fine-tuning

In [ ]:
torch.manual_seed(0)  # the new output layer starts random; a fixed seed makes this reproducible
model_before = build_classifier()  # downloads the ImageNet weights on first use
loss0, acc0 = evaluate(model_before, val_x, val_y)
print(f"ImageNet ResNet-18 with a new 10-class layer, before any training: validation loss {loss0:.3f} "
      f"(ln 10 = {np.log(10):.3f}), accuracy {acc0:.3f}")
del model_before

## Fine-tune

In [ ]:
t = time.time()
torch.cuda.reset_peak_memory_stats()
classifier, history, step_losses, best_epoch = train_classifier(train_x, train_y, {"validation": (val_x, val_y)},
                                                                 epochs=CLS_EPOCHS, seed=0)
TIMES["classifier fine-tuning"] = time.time() - t
memory_report("classifier fine-tuning")
print(f"kept epoch {best_epoch}; {len(step_losses)} optimisation steps in {TIMES['classifier fine-tuning']:.0f} s")

## The loss decreases

Left: the training loss of every optimisation step (grey) and its running mean over 10 steps. Right: the loss and
accuracy after each epoch on the training batches (augmented) and on the validation set.

In FAST mode an epoch is only 10 steps, so the **validation** loss can jump up in the first epochs before it falls: the
network is evaluated in eval mode, which uses batch-norm *running* statistics, and these lag behind weights that change
quickly while the learning rate is at its peak. Once the learning rate decays the two agree again. With the full
settings (97 steps per epoch) this effect is small (Tutorial 3 shows the full runs' curves).

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
steps = np.arange(1, len(step_losses) + 1)
axes[0].plot(steps, step_losses, color="#c3c2b7", lw=1, label="each step")
run = np.convolve(step_losses, np.ones(10) / 10, mode="valid")
axes[0].plot(np.arange(10, len(step_losses) + 1), run, color="#2a78d6", lw=2, label="mean of 10 steps")
axes[0].axhline(np.log(10), color="gray", ls=":", lw=1)
axes[0].text(len(step_losses), np.log(10), "ln 10 (random guess)", ha="right", va="bottom", fontsize=8, color="gray")
axes[0].set_xlabel("optimisation step")
axes[0].set_ylabel("cross-entropy loss")
axes[0].legend(fontsize=8)
ep = [0] + [h["epoch"] for h in history]
axes[1].plot(ep, [loss0] + [h["validation_loss"] for h in history], "o-", color="#2a78d6", label="validation")
axes[1].plot(ep[1:], [h["train_loss"] for h in history], "s:", color="#2a78d6", label="training (during the epoch)")
axes[1].set_ylabel("cross-entropy loss")
axes[2].plot(ep, [100 * acc0] + [100 * h["validation_accuracy"] for h in history], "o-", color="#2a78d6",
             label="validation")
axes[2].plot(ep[1:], [100 * h["train_accuracy"] for h in history], "s:", color="#2a78d6", label="training (during the epoch)")
axes[2].set_ylabel("accuracy (%)")
for ax in axes[1:]:
    ax.set_xticks(ep)
    ax.set_xlabel("epoch (0 = before fine-tuning)")
    ax.legend(fontsize=8)
for ax in axes:
    ax.grid(alpha=0.3)
fig.suptitle("Classifier fine-tuning: the loss decreases", x=0.01, ha="left")
plt.tight_layout()
plt.show()
print(f"first 10 steps: mean loss {np.mean(step_losses[:10]):.3f}; last 10 steps: {np.mean(step_losses[-10:]):.3f}")

## Final result: the released classifier

The same loop with the full settings (all 12,330 images, 30 epochs) produced the released classifier. Loading it and
scoring the same validation images shows where the fine-tuning ends up.

In [ ]:
from safetensors.torch import load_file

released = build_classifier(pretrained=False)
released.load_state_dict(load_file(WEIGHTS_DIR / "resnet18_real_only_seed0.safetensors"))
fast_loss, fast_acc = evaluate(classifier, val_x, val_y)
rel_loss, rel_acc = evaluate(released, val_x, val_y)
print(f"{'':<34}{'validation loss':>16}{'validation accuracy':>21}")
print(f"{'before fine-tuning':<34}{loss0:>16.3f}{acc0:>21.3f}")
print(f"{'this run (' + ('FAST' if FAST else 'full') + ')':<34}{fast_loss:>16.3f}{fast_acc:>21.3f}")
print(f"{'released (full settings, paper)':<34}{rel_loss:>16.3f}{rel_acc:>21.3f}")
del released

# Part 2: Fine-tuning the generator (LoRA)

## The pretrained model

The generator starts from **Stable Diffusion 1.5** (`stable-diffusion-v1-5/stable-diffusion-v1-5` on Hugging Face; the VAE and U-Net in 16-bit, 1.9 GB,
downloaded on first use). Its 860-million-parameter U-Net stays frozen. What is trained:

- **LoRA adapters** (rank 8) on the attention projections `to_q`, `to_k`, `to_v`, `to_out.0` of all 16 transformer
  blocks: 1,594,368 values, initialised so that the adapters start as a no-op (their `B` matrices are zero);
- a **class-embedding table** (11 x 77 x 768) that replaces the text prompt: it starts as CLIP's encoding of prompts
  such as "a telescope image of a galaxy, barred spiral morphology", plus the empty prompt as the null class (the CLIP
  text encoder is downloaded once for this and then discarded).

**The optimization loop** is the project's `diffusion/train_lora.py`: encode a batch of training images (flipped and
rotated at random, upsampled to 512 px) into latents with the VAE; replace the class by the null class with
probability 0.1 (so the model also learns unconditional predictions, needed for classifier-free guidance); add noise
at a random level; predict the noise; mean squared error; backpropagate through the frozen U-Net into the LoRA weights
and the table; clip the gradient norm to 1; AdamW step (learning rate 1e-4, weight decay 0.01, linear warm-up then
constant).

Because the training loss mostly measures *which noise level was drawn* (Tutorial 3), it is noisy and nearly flat.
To see the model change, the loop also computes the loss on a **fixed evaluation batch**: one validation image per
class, with fixed noise and fixed noise levels from low to high.

In [ ]:
import json

from diffusers import AutoencoderKL, DDPMScheduler, DPMSolverMultistepScheduler, UNet2DConditionModel
from diffusers.optimization import get_scheduler
from peft import LoraConfig
from peft.utils import set_peft_model_state_dict
from safetensors.torch import load_file
from transformers import CLIPTextModel, CLIPTokenizer

SD_REPO = "stable-diffusion-v1-5/stable-diffusion-v1-5"
LORA_CONFIG = json.loads((WEIGHTS_DIR / "lora_config.json").read_text())  # the released run's settings


def load_base_generator():
    """Stable Diffusion 1.5's VAE and U-Net (16-bit files from Hugging Face, frozen) and its training noise schedule."""
    vae = AutoencoderKL.from_pretrained(SD_REPO, subfolder="vae", variant="fp16", torch_dtype=AMP_DTYPE).to(DEVICE)
    unet = UNet2DConditionModel.from_pretrained(SD_REPO, subfolder="unet", variant="fp16",
                                                torch_dtype=AMP_DTYPE).to(DEVICE)
    vae.requires_grad_(False)
    unet.requires_grad_(False)
    vae.eval()
    return vae, unet, DDPMScheduler.from_pretrained(SD_REPO, subfolder="scheduler")


def add_lora(unet, cfg=LORA_CONFIG):
    """Add LoRA adapters (rank 8 on to_q/to_k/to_v/to_out.0); their weights are kept in fp32."""
    unet.add_adapter(LoraConfig(r=cfg["lora_rank"], lora_alpha=cfg["lora_alpha"], init_lora_weights="gaussian",
                                target_modules=cfg["lora_target_modules"]))
    params = [p for p in unet.parameters() if p.requires_grad]
    for p in params:
        p.data = p.data.float()
    return params


@torch.no_grad()
def clip_class_table(cfg=LORA_CONFIG):
    """Initial class-embedding table: CLIP's encoding of one prompt per class + the empty prompt (the null class)."""
    tokenizer = CLIPTokenizer.from_pretrained(SD_REPO, subfolder="tokenizer")
    encoder = CLIPTextModel.from_pretrained(SD_REPO, subfolder="text_encoder", variant="fp16",
                                            torch_dtype=torch.float32).to(DEVICE)
    tokens = tokenizer(cfg["class_prompts"] + [""], padding="max_length", max_length=tokenizer.model_max_length,
                       truncation=True, return_tensors="pt").input_ids.to(DEVICE)
    table = encoder(tokens)[0].float()
    del encoder
    torch.cuda.empty_cache()
    return table  # [11, 77, 768]


def to_vae_input(images_u8, augment, resolution=512):
    """uint8 NHWC 256 px -> float NCHW in [-1, 1] at 512 px (preprocess in diffusion/train_lora.py)."""
    x = images_u8.permute(0, 3, 1, 2).float().div_(127.5).sub_(1.0)
    if augment:
        flip = torch.rand(x.shape[0], device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(-1), x)
        k = torch.randint(0, 4, (x.shape[0],), device=x.device)
        x = torch.stack([torch.rot90(xi, int(ki), dims=(-2, -1)) for xi, ki in zip(x, k)])
    return F.interpolate(x, size=(resolution, resolution), mode="bicubic", align_corners=False).clamp_(-1.0, 1.0)


def fixed_eval_batch(vae, noise_scheduler, x_u8, y, seed=0):
    """A fixed evaluation batch: latents of held-out images with fixed noise and fixed noise levels, so the
    diffusion loss on it changes only when the model changes (the training loss is noisy by design)."""
    g = torch.Generator(DEVICE).manual_seed(seed)
    with torch.no_grad():
        latents = vae.encode(to_vae_input(x_u8.to(DEVICE), False).to(AMP_DTYPE)).latent_dist.mean.float() \
            * vae.config.scaling_factor
    noise = torch.randn(latents.shape, generator=g, device=DEVICE)
    t = torch.linspace(50, 950, len(latents), device=DEVICE).long()
    return {"noisy": noise_scheduler.add_noise(latents, noise, t), "noise": noise, "t": t, "y": y.to(DEVICE)}


@torch.no_grad()
def eval_diffusion_loss(unet, table, batch):
    with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
        pred = unet(batch["noisy"].to(AMP_DTYPE), batch["t"],
                    encoder_hidden_states=table[batch["y"]].to(AMP_DTYPE)).sample
    return F.mse_loss(pred.float(), batch["noise"]).item()


def train_lora(unet, vae, noise_scheduler, lora_params, table, train_x, train_y, steps, batch_size, eval_batch,
               eval_every, learning_rate=1e-4, weight_decay=0.01, warmup_steps=500, cond_dropout=0.1,
               max_grad_norm=1.0, seed=42):
    """The project's LoRA training loop (diffusion/train_lora.py): AdamW on the LoRA weights + class table, constant
    learning rate after a linear warm-up, classifier-free-guidance dropout, MSE on the predicted noise, gradient
    clipping. Returns the per-step training losses and the fixed-batch evaluation losses [(step, loss)]."""
    torch.manual_seed(seed)
    table.requires_grad_(True)
    optimizer = torch.optim.AdamW([{"params": lora_params}, {"params": [table]}], lr=learning_rate,
                                  weight_decay=weight_decay)
    scheduler = get_scheduler("constant_with_warmup", optimizer=optimizer, num_warmup_steps=warmup_steps,
                              num_training_steps=steps)
    scaler = torch.amp.GradScaler("cuda", enabled=USE_GRAD_SCALER)
    null = table.shape[0] - 1
    step_losses, eval_losses = [], [(0, eval_diffusion_loss(unet, table, eval_batch))]
    unet.train()
    perm, pos, t0 = torch.randperm(len(train_x)), 0, time.time()
    for step in range(1, steps + 1):
        if pos + batch_size > len(perm):
            perm, pos = torch.randperm(len(train_x)), 0
        bi = perm[pos:pos + batch_size]
        pos += batch_size
        with torch.no_grad():
            pixels = to_vae_input(train_x[bi].to(DEVICE), augment=True).to(AMP_DTYPE)
            latents = vae.encode(pixels).latent_dist.sample().float() * vae.config.scaling_factor
        labels = train_y[bi].to(DEVICE)
        drop = torch.rand(len(labels), device=DEVICE) < cond_dropout  # classifier-free guidance training
        labels = torch.where(drop, torch.full_like(labels, null), labels)
        noise = torch.randn_like(latents)
        t = torch.randint(0, noise_scheduler.config.num_train_timesteps, (len(latents),), device=DEVICE)
        noisy = noise_scheduler.add_noise(latents, noise, t)
        with torch.autocast("cuda", dtype=AMP_DTYPE, enabled=AMP_DTYPE != torch.float32):
            pred = unet(noisy.to(AMP_DTYPE), t, encoder_hidden_states=table[labels]).sample
        loss = F.mse_loss(pred.float(), noise)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(lora_params + [table], max_grad_norm)
        scaler.step(optimizer)
        scaler.update()
        scheduler.step()
        optimizer.zero_grad(set_to_none=True)
        step_losses.append(loss.item())
        if step % eval_every == 0 or step == steps:
            unet.eval()
            eval_losses.append((step, eval_diffusion_loss(unet, table, eval_batch)))
            unet.train()
            print(f"  step {step:>5}/{steps}  training loss (mean of the last {eval_every}) "
                  f"{np.mean(step_losses[-eval_every:]):.4f}  fixed-batch loss {eval_losses[-1][1]:.4f}  "
                  f"({time.time() - t0:.0f} s)")
    unet.eval()
    table.requires_grad_(False)
    return step_losses, eval_losses


def load_released_lora(unet):
    """Put the released LoRA weights into the U-Net's adapters; returns the released class table."""
    result = set_peft_model_state_dict(unet, load_file(WEIGHTS_DIR / "lora_unet.safetensors"))
    assert not result.unexpected_keys and not [k for k in result.missing_keys if "lora" in k]
    return load_file(WEIGHTS_DIR / "class_embeddings.safetensors")["table"].to(DEVICE)


@torch.no_grad()
def generate(unet, vae, table, labels, seeds, guidance=3.0, steps=30, resolution=512, batch_size=10):
    """Class-conditional images (Tutorial 4's sampling loop, batched): classifier-free guidance with the null class,
    30 DPM-Solver++ steps, VAE decode, Lanczos down to 256 px. One noise generator per image, as in the project."""
    scheduler = DPMSolverMultistepScheduler.from_pretrained(SD_REPO, subfolder="scheduler")
    null, images = table.shape[0] - 1, []
    for s in range(0, len(labels), batch_size):
        lab = torch.as_tensor(labels[s:s + batch_size], device=DEVICE)
        latents = torch.cat([torch.randn((1, 4, resolution // 8, resolution // 8), device=DEVICE, dtype=AMP_DTYPE,
                                         generator=torch.Generator(DEVICE).manual_seed(int(sd)))
                             for sd in seeds[s:s + batch_size]])
        scheduler.set_timesteps(steps, device=DEVICE)
        latents = latents * scheduler.init_noise_sigma
        cond = torch.cat([table[torch.full_like(lab, null)], table[lab]]).to(AMP_DTYPE)
        for t in scheduler.timesteps:
            inp = scheduler.scale_model_input(torch.cat([latents] * 2), t)
            eps_null, eps_class = unet(inp, t, encoder_hidden_states=cond).sample.chunk(2)
            latents = scheduler.step(eps_null + guidance * (eps_class - eps_null), t, latents).prev_sample
        x = vae.decode(latents / vae.config.scaling_factor).sample
        x = ((x.float() / 2 + 0.5).clamp(0, 1) * 255).round().byte().permute(0, 2, 3, 1).cpu().numpy()
        images += [np.asarray(Image.fromarray(im).resize((256, 256), Image.LANCZOS)) for im in x]
    return images

In [ ]:
t = time.time()
torch.cuda.reset_peak_memory_stats()
vae, unet, noise_scheduler = load_base_generator()
lora_params = add_lora(unet)
class_table = clip_class_table()
eval_items = [next(item for item in zip_members(dataset_zip, "validation") if item[1] == c) for c in range(10)]
eval_x, eval_y = load_images(dataset_zip, eval_items)
eval_batch = fixed_eval_batch(vae, noise_scheduler, eval_x, eval_y)
TIMES["load generator"] = time.time() - t
print(f"U-Net {sum(p.numel() for p in unet.parameters()) - sum(p.numel() for p in lora_params):,} frozen parameters; "
      f"LoRA {sum(p.numel() for p in lora_params):,} + class table {class_table.numel():,} trained")
print(f"loaded in {TIMES['load generator']:.0f} s")

## Before fine-tuning

With the LoRA adapters still a no-op, the model is plain Stable Diffusion 1.5 steered by CLIP's encoding of the class
prompts. The paper's seeds for sample 0 of four classes are used, so the same noise is used before and after training.

In [ ]:
t = time.time()
images_before = generate(unet, vae, class_table, GEN_CLASSES, GEN_SEEDS)
TIMES["generate"] = time.time() - t
print(f"generated {len(images_before)} images in {TIMES['generate']:.0f} s")

## Fine-tune

In [ ]:
t = time.time()
torch.cuda.reset_peak_memory_stats()
lora_losses, eval_losses = train_lora(unet, vae, noise_scheduler, lora_params, class_table, train_x, train_y,
                                      steps=LORA_STEPS, batch_size=LORA_BATCH, eval_batch=eval_batch,
                                      eval_every=LORA_EVAL_EVERY, warmup_steps=LORA_WARMUP)
TIMES["LoRA fine-tuning"] = time.time() - t
memory_report("LoRA fine-tuning")
print(f"{LORA_STEPS} steps in {TIMES['LoRA fine-tuning']:.0f} s ({TIMES['LoRA fine-tuning'] / LORA_STEPS:.2f} s/step)")

## The loss decreases

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.4))
axes[0].plot(np.arange(1, len(lora_losses) + 1), lora_losses, color="#c3c2b7", lw=1, label="each step")
w = min(25, len(lora_losses))
axes[0].plot(np.arange(w, len(lora_losses) + 1), np.convolve(lora_losses, np.ones(w) / w, mode="valid"),
             color="#eb6834", lw=2, label=f"mean of {w} steps")
axes[0].set_xlabel("step")
axes[0].set_ylabel("MSE loss")
axes[0].set_title("training loss (random images, noise levels)", loc="left", fontsize=10)
axes[0].legend(fontsize=8)
axes[1].plot([s for s, _ in eval_losses], [l for _, l in eval_losses], "o-", color="#eb6834")
axes[1].set_xlabel("step")
axes[1].set_ylabel("MSE loss")
axes[1].set_title("fixed evaluation batch (same images, noise, levels)", loc="left", fontsize=10)
for ax in axes:
    ax.grid(alpha=0.3)
fig.suptitle("LoRA fine-tuning of the generator", x=0.01, ha="left")
plt.tight_layout()
plt.show()
print(f"fixed-batch loss: {eval_losses[0][1]:.4f} before -> {eval_losses[-1][1]:.4f} after {LORA_STEPS} steps "
      f"({100 * (eval_losses[-1][1] / eval_losses[0][1] - 1):+.1f}%)")

## Final result: before, after, and the released generator

The same seeds, generated after the FAST fine-tuning, and with the **released** LoRA weights and class table (10,000
steps at batch 8): the last row reproduces sample 0 of these classes in the paper's synthetic training set.

In [ ]:
t = time.time()
images_after = generate(unet, vae, class_table, GEN_CLASSES, GEN_SEEDS)
released_table = load_released_lora(unet)  # replaces the adapters' weights with the released ones
images_released = generate(unet, vae, released_table, GEN_CLASSES, GEN_SEEDS)
TIMES["generate"] += time.time() - t
memory_report("generation")

rows = [("before fine-tuning", images_before), (f"after {LORA_STEPS} steps ({'FAST' if FAST else 'full'})", images_after),
        ("released (10,000 steps)", images_released)]
fig, axes = plt.subplots(3, len(GEN_CLASSES), figsize=(2.6 * len(GEN_CLASSES), 8.2))
for r, (label, imgs) in enumerate(rows):
    for c, img in enumerate(imgs):
        axes[r, c].imshow(img)
        axes[r, c].set_xticks([])
        axes[r, c].set_yticks([])
        if r == 0:
            axes[r, c].set_title(CLASS_NAMES[GEN_CLASSES[c]], fontsize=9)
    axes[r, 0].set_ylabel(label, fontsize=9)
plt.tight_layout()
plt.show()

# Summary

Both pretrained models were fine-tuned with the project's loops: the classifier's loss fell from ln 10 towards the
level of the released model within a few epochs, and the generator's fixed-batch loss fell while its images moved from
generic Stable Diffusion galaxies towards the survey's look. Tutorial 6 runs the whole pipeline (generator,
synthetic data, classifiers on real / mixed / synthetic data) and follows the losses until they converge.

In [ ]:
TIMES["whole notebook"] = time.time() - T0
for step, seconds in TIMES.items():
    print(f"  {step:<26} {seconds / 60:5.1f} min")
print("peak GPU memory:", {k: f"{v:.1f} GiB" for k, v in MEMORY.items()})